# 06. Building Your First Complete Agent: The Beginner Capstone

Welcome to the capstone of the beginner curriculum. We are going to build **one complete, bounded, testable agent** using the hybrid architecture typical of real enterprise systems.

**Scenario:** A customer escalated ticket T-102: "I was charged twice for my subscription. Support has not resolved this. Please fix it."

The agent must review the ticket, inspect billing, propose a resolution, and if a refund is needed, request human approval before executing it safely.

## Part 1: Define the Domain Fixtures
First, we create deterministic fixtures simulating a real business database.

In [1]:
import json
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('Northstar')

DB = {
    'tickets': {'T-102': {'customer_id': 'C-55', 'text': 'I was charged twice for my subscription.', 'status': 'open'}},
    'customers': {'C-55': {'name': 'Alice', 'plan': 'Pro'}},
    'billing': {'C-55': {'status': 'active', 'card_valid': True}},
    'transactions': {'C-55': [
        {'tx_id': 'TX-901', 'amount_cents': 10000, 'date': '2026-08-01'},
        {'tx_id': 'TX-902', 'amount_cents': 10000, 'date': '2026-08-01', 'note': 'system duplicate'}
    ]},
    'refunds': []
}

print('Domain fixtures initialized.')

Domain fixtures initialized.


## Part 2: Define Trusted Context
The LLM does not invent its own identity. We define it cryptographically/statically via the application.

In [2]:
from pydantic import BaseModel, Field
from typing import Literal, Any, Optional

class ExecutionContext(BaseModel):
    user_id: str
    tenant_id: str
    roles: set[str]
    request_id: str

ctx = ExecutionContext(user_id='U-88', tenant_id='Northstar', roles={'support:agent', 'refund:request'}, request_id='REQ-999')
print(f'Running as: {ctx}')

Running as: user_id='U-88' tenant_id='Northstar' roles={'refund:request', 'support:agent'} request_id='REQ-999'


## Part 3 & 4: Tool I/O Models & Read-Only Tools
Tools must be strictly typed. We define them as isolated functions.

In [3]:
class GetTicketArgs(BaseModel):
    ticket_id: str

def get_ticket_details(args: GetTicketArgs) -> str:
    """Fetches the text of a support ticket."""
    if args.ticket_id not in DB['tickets']:
        return 'Error: Ticket not found'
    return json.dumps(DB['tickets'][args.ticket_id])

class GetCustomerArgs(BaseModel):
    customer_id: str

def get_recent_transactions(args: GetCustomerArgs) -> str:
    """Fetches recent transactions for a customer."""
    if args.customer_id not in DB['transactions']:
        return 'Error: No transactions'
    return json.dumps(DB['transactions'][args.customer_id])

def get_refund_policy(args: BaseModel) -> str:
    """Fetches the company refund policy."""
    return 'POLICY: Duplicate charges must be refunded in full. Refunds require manager approval.'


## Part 5: Define the Refund Proposal
When the agent decides an action is needed, it produces a *Proposal*, not an immediate side effect.

In [4]:
class RefundProposal(BaseModel):
    customer_id: str
    transaction_id: str
    amount_cents: int
    reason: str


## Part 6: Write Tool + Idempotency
The consequential action. Notice the `idempotency_key` preventing double refunds if retried.

In [5]:
class IssueRefundArgs(BaseModel):
    customer_id: str
    transaction_id: str
    amount_cents: int
    idempotency_key: str

def issue_refund(args: IssueRefundArgs) -> str:
    """Issues a financial refund."""
    # Idempotency check
    for r in DB['refunds']:
        if r['idempotency_key'] == args.idempotency_key:
            return f'Refund already processed for {args.idempotency_key}'
    
    # Execution
    DB['refunds'].append(args.model_dump())
    return f'Refund of {args.amount_cents} cents issued successfully for TX {args.transaction_id}'


## Part 7: Tool Registry & Permissions
We define exactly which tools exist and their authorization requirements.

In [6]:
from typing import Callable

class ToolDefinition(BaseModel):
    name: str
    effect: Literal['READ_ONLY', 'CONSEQUENTIAL_WRITE']
    required_permission: str
    func: Callable
    schema: type[BaseModel]

TOOL_REGISTRY = {
    'get_ticket_details': ToolDefinition(name='get_ticket_details', effect='READ_ONLY', required_permission='support:read', func=get_ticket_details, schema=GetTicketArgs),
    'get_recent_transactions': ToolDefinition(name='get_recent_transactions', effect='READ_ONLY', required_permission='billing:read', func=get_recent_transactions, schema=GetCustomerArgs),
    'get_refund_policy': ToolDefinition(name='get_refund_policy', effect='READ_ONLY', required_permission='support:read', func=get_refund_policy, schema=BaseModel),
    'issue_refund': ToolDefinition(name='issue_refund', effect='CONSEQUENTIAL_WRITE', required_permission='refund:issue', func=issue_refund, schema=IssueRefundArgs),
}

AUTONOMOUS_TOOLS = ['get_ticket_details', 'get_recent_transactions', 'get_refund_policy']


/var/folders/h9/tj11chyd3w12p_kly5_d3dsw0000gn/T/ipykernel_67013/761109170.py:3: UserWarning: Field name "schema" in "ToolDefinition" shadows an attribute in parent "BaseModel"
  class ToolDefinition(BaseModel):


## Part 8 & 9: Model Contract & Deterministic Stub
We define the shape of the LLM's response, and create a mock model to ensure the core lab runs without an API key.

In [7]:
class ToolCall(BaseModel):
    id: str
    name: str
    arguments: dict

class AgentDecision(BaseModel):
    tool_calls: list[ToolCall] = []
    proposal: Optional[RefundProposal] = None
    final_answer: Optional[str] = None

class MockDecisionModel:
    """Deterministic stub for reproducible training. Returns fixed responses based on the turn."""
    def __init__(self):
        self.turn = 0
    def decide(self, state) -> AgentDecision:
        self.turn += 1
        if self.turn == 1:
            return AgentDecision(tool_calls=[ToolCall(id='call_1', name='get_ticket_details', arguments={'ticket_id': state.ticket_id})])
        if self.turn == 2:
            return AgentDecision(tool_calls=[
                ToolCall(id='call_2', name='get_recent_transactions', arguments={'customer_id': 'C-55'}),
                ToolCall(id='call_3', name='get_refund_policy', arguments={})
            ])
        if self.turn == 3:
            # Identifies duplicate TX-902 and proposes refund
            return AgentDecision(proposal=RefundProposal(
                customer_id='C-55', transaction_id='TX-902', amount_cents=10000, reason='Duplicate charge identified.'
            ))
        return AgentDecision(final_answer='I am stuck.')


## Part 10: The Complete Bounded Loop
Here we implement state, budgets, no-progress detection, schema validation, and authorization.

In [8]:
class AgentState(BaseModel):
    ticket_id: str
    steps: int = 0
    max_steps: int = 5
    history: list[dict] = []
    seen_actions: set[str] = set()
    terminal_reason: Optional[str] = None
    proposal: Optional[RefundProposal] = None

def run_agent(ctx: ExecutionContext, state: AgentState, model) -> AgentState:
    while state.terminal_reason is None:
        if state.steps >= state.max_steps:
            state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
            break
        
        state.steps += 1
        decision = model.decide(state)
        
        if decision.final_answer:
            state.terminal_reason = 'SUCCESS'
            break
            
        if decision.proposal:
            state.proposal = decision.proposal
            state.terminal_reason = 'HUMAN_APPROVAL_REQUIRED'
            break
            
        for tc in decision.tool_calls:
            # 1. No-progress detection
            fingerprint = f'{tc.name}:{tc.arguments}'
            if fingerprint in state.seen_actions:
                state.terminal_reason = 'NO_PROGRESS'
                break
            state.seen_actions.add(fingerprint)
            
            # 2. Registry & Autonomous Auth
            if tc.name not in AUTONOMOUS_TOOLS:
                state.terminal_reason = 'AUTHORIZATION_DENIED'
                break
            
            tdef = TOOL_REGISTRY[tc.name]
            
            # 3. Schema Validation & Execution
            try:
                args_obj = tdef.schema(**tc.arguments)
                result = tdef.func(args_obj)
                state.history.append({'tool': tc.name, 'result': result})
            except Exception as e:
                state.history.append({'tool': tc.name, 'error': str(e)})
    return state


## Part 11 & 12: Happy Path & Human Approval
Run the agent to get a proposal, then securely bind human approval to it.

In [9]:
import hashlib

state = AgentState(ticket_id='T-102')
model = MockDecisionModel()
final_state = run_agent(ctx, state, model)

print(f'Terminal Reason: {final_state.terminal_reason}')
print(f'Proposal: {final_state.proposal}')

class Approval(BaseModel):
    proposal_digest: str
    approver_id: str
    decision: Literal['approve', 'reject']

digest = hashlib.sha256(final_state.proposal.model_dump_json().encode()).hexdigest()
manager_approval = Approval(proposal_digest=digest, approver_id='MGR-1', decision='approve')
print(f'Manager approved digest: {digest[:8]}...')


Terminal Reason: HUMAN_APPROVAL_REQUIRED
Proposal: customer_id='C-55' transaction_id='TX-902' amount_cents=10000 reason='Duplicate charge identified.'
Manager approved digest: fe3c005f...


## Part 13 & 14: Execution & Idempotency
We execute the approved proposal using a secure idempotency key. Then we try executing it again to prove safety.

In [10]:
def execute_approved_proposal(proposal: RefundProposal, approval: Approval):
    check_digest = hashlib.sha256(proposal.model_dump_json().encode()).hexdigest()
    if check_digest != approval.proposal_digest or approval.decision != 'approve':
        raise ValueError('Invalid or rejected approval.')
        
    # Business Validation: Ensure refund doesn't exceed TX amount
    # (Simplified for demo)
    
    # Generate Idempotency Key bound to this exact request
    idem_key = f'ref_{proposal.transaction_id}_{check_digest[:8]}'
    args = IssueRefundArgs(
        customer_id=proposal.customer_id, 
        transaction_id=proposal.transaction_id,
        amount_cents=proposal.amount_cents,
        idempotency_key=idem_key
    )
    return issue_refund(args)

print('First Execution:', execute_approved_proposal(final_state.proposal, manager_approval))
print('Duplicate Execution:', execute_approved_proposal(final_state.proposal, manager_approval))
print('DB State:', DB['refunds'])


First Execution: Refund of 10000 cents issued successfully for TX TX-902
Duplicate Execution: Refund already processed for ref_TX-902_fe3c005f
DB State: [{'customer_id': 'C-55', 'transaction_id': 'TX-902', 'amount_cents': 10000, 'idempotency_key': 'ref_TX-902_fe3c005f'}]


## Part 18: Real OpenAI Integration (Optional)
If you provide an `OPENAI_API_KEY` in your environment, we can run the EXACT SAME application boundary using a real LLM. We define an `OpenAIDecisionModel` that adheres to our `DecisionModel` contract.

In [11]:
import os
from openai import OpenAI

class OpenAIDecisionModel:
    def __init__(self):
        self.client = OpenAI()
        self.messages = [
            {'role': 'system', 'content': 'You are a support agent. Analyze the ticket, query billing, and if there is a duplicate charge, use the propose_refund tool.'}
        ]
        # We map our ToolRegistry to OpenAI Schema natively here
        self.tools = [
            {'type': 'function', 'function': {'name': 'get_ticket_details', 'parameters': {'type': 'object', 'properties': {'ticket_id': {'type': 'string'}}, 'required': ['ticket_id']}}},
            {'type': 'function', 'function': {'name': 'get_recent_transactions', 'parameters': {'type': 'object', 'properties': {'customer_id': {'type': 'string'}}, 'required': ['customer_id']}}},
            {'type': 'function', 'function': {'name': 'get_refund_policy', 'parameters': {'type': 'object', 'properties': {}}}},
            {'type': 'function', 'function': {'name': 'propose_refund', 'parameters': RefundProposal.model_json_schema()}}
        ]
        
    def decide(self, state: AgentState) -> AgentDecision:
        # Feed observation history into LLM
        for h in state.history:
            self.messages.append({'role': 'system', 'content': f'Observation: {h}'})
        state.history.clear() # clear unread
        
        if len(self.messages) == 1:
            self.messages.append({'role': 'user', 'content': f'Investigate ticket {state.ticket_id}'})
            
        resp = self.client.chat.completions.create(model='gpt-4o-mini', messages=self.messages, tools=self.tools)
        msg = resp.choices[0].message
        self.messages.append(msg)
        
        if not msg.tool_calls:
            return AgentDecision(final_answer=msg.content)
            
        tcs = []
        for tc in msg.tool_calls:
            if tc.function.name == 'propose_refund':
                return AgentDecision(proposal=RefundProposal(**json.loads(tc.function.arguments)))
            tcs.append(ToolCall(id=tc.id, name=tc.function.name, arguments=json.loads(tc.function.arguments)))
        return AgentDecision(tool_calls=tcs)

if os.getenv('OPENAI_API_KEY'):
    print('Running REAL OpenAI Model through the secure runtime...')
    real_state = AgentState(ticket_id='T-102')
    real_model = OpenAIDecisionModel()
    final_real_state = run_agent(ctx, real_state, real_model)
    print(f'Terminal Reason: {final_real_state.terminal_reason}')
    print(f'Proposal: {final_real_state.proposal}')
else:
    print('Skipping real execution. Set OPENAI_API_KEY in environment to run.')

Skipping real execution. Set OPENAI_API_KEY in environment to run.


## Part 19: Optional Framework Mapping
As you saw in Module 05, frameworks like OpenAI Agents SDK, LangGraph, or PydanticAI exist to package the `run_agent` while loop boilerplate. However, they **do not** replace the Business Validation, Schema Validation, Authorization, or Idempotency logic we just built. The framework is just the orchestrator; your application boundary is the security.